# ML-прогноз золото-урановых рудных узлов

Исправленная версия ноутбука: `geo_score` используется только как baseline для сравнения, но не входит в признаки ML и не доминирует в итоговом прогнозе.

Ключевые изменения:
- убран `geo_score_sm` из `feature_cols`;
- добавлена пространственная валидация `GroupKFold`;
- добавлены метрики `ROC-AUC`, `Average Precision`, `Recall@Top area`, `Lift@Top area`;
- отрицательные примеры выбираются аккуратно, а не все неизвестные ячейки считаются нулём;
- итоговый слой `ml_prospectivity` строится от ML-модели;
- `prognoz = 1 - ml_prospectivity` оставлен для совместимости с блоком «Прогноз».

In [1]:
import json
import os
import re
import shutil
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union, nearest_points
from shapely.prepared import prep

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

# =========================================================
# 1. SETTINGS
# =========================================================
CELL_SIZE = 500
RANDOM_STATE = 42

# ML-настройки
MODEL_KIND = "RandomForest"  # "RandomForest" или "ExtraTrees"
RF_N_ESTIMATORS = 700
RF_MAX_DEPTH = 10
RF_MIN_SAMPLES_LEAF = 5
RF_MIN_SAMPLES_SPLIT = 10

# Пространственная валидация: размер блока в ячейках.
# 8 означает 8*500 = 4 км.
SPATIAL_BLOCK_CELLS = 8
N_SPLITS = 5

# Positive-unlabeled / negative sampling
# Неизвестные ячейки не считаются автоматически отрицательными.
NEGATIVE_TO_POSITIVE_RATIO = 4
MIN_POSITIVE_CELLS = 10
MAX_DISTANCE_TO_POSITIVE_FOR_NEGATIVE = None  # можно задать, например, 5000; None = не использовать

# Для pseudo-negative: берём ячейки с низким expert baseline и далеко от известных точек.
NEGATIVE_GEO_QUANTILE = 0.45
NEGATIVE_DISTANCE_QUANTILE = 0.55

# Итоговая модель: ML в основе.
# geo_score не входит в признаки ML. Ниже он используется только как слабая регуляризация/сравнение.
FINAL_W_ML = 0.80
FINAL_W_COINCIDENCE = 0.15
FINAL_W_LOCAL = 0.05

# Если известных точек мало и ML ненадёжен, можно включить fallback к экспертному baseline.
ALLOW_EXPERT_FALLBACK_IF_TOO_FEW_POINTS = True

# Преобразования расстояний в близость
Q_FACIES = 0.78
Q_PALEO = 0.76
Q_STRUCT = 0.72
Q_MAGM = 0.42
Q_TECT1 = 0.74
Q_TECT2 = 0.74

TOP_AREA_LEVELS = [0.01, 0.03, 0.05, 0.10, 0.15]

# =========================================================
# 2. PATHS
# =========================================================
def find_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data/Прогноз"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base

    # Если передан zip рядом с ноутбуком, попробуем распаковать автоматически.
    for zip_path in [Path("/mnt/data/Прогноз.zip"), Path.cwd() / "Прогноз.zip", Path.cwd() / "Prognoz.zip"]:
        if zip_path.exists():
            import zipfile
            out = Path("/mnt/data/prog_zip")
            out.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(out)
            for p in [out, *out.glob("**")]:
                if (p / "shp_dbf" / "svita_new.shp").exists():
                    return p
    raise FileNotFoundError("Не найден каталог с shp_dbf/svita_new.shp. Положите рядом папку проекта или Прогноз.zip.")

BASE_DIR = find_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "ml_forecast_result"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAFE_ALIAS_DIR = OUT_DIR / "_safe_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "ml_forecast_result.gpkg"
OUT_PNG = OUT_DIR / "ml_forecast_map.png"
OUT_COMPARE = OUT_DIR / "ml_vs_baseline_compare.png"
OUT_IMPORTANCE = OUT_DIR / "feature_importance.png"
OUT_CSV = OUT_DIR / "ml_forecast_grid.csv"
OUT_JSON = OUT_DIR / "ml_metrics.json"

print("BASE_DIR:", BASE_DIR)
print("SHP_DIR:", SHP_DIR)
print("OUT_DIR:", OUT_DIR)

# =========================================================
# 3. HELPERS
# =========================================================
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    if np.isclose(mx, mn):
        return np.full_like(arr, 0.5, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def robust_normalize_01(values, q_low=0.03, q_high=0.97):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        return normalize_01(arr)
    return np.clip((arr - lo) / (hi - lo), 0, 1)

def safe_quantile(values, q, default=0.0):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return default
    return float(np.quantile(arr, q))

def smooth_on_regular_grid(grid, value_col, shape, passes=1):
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()
    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()
    kernel = np.array([[1.0, 1.2, 1.0], [1.2, 3.0, 1.2], [1.0, 1.2, 1.0]], dtype=float)
    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        smoothed = np.where(den > 0, num / den, np.nan)
    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]

def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None

def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases, stems = {}, {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")) or name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_-. ") for ch in base_s)
        except Exception:
            safe = False
            base_s = None

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias = f"layer_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias}_shp.pj4")
        aliases[alias] = alias_dir / f"{alias}.shp"
    return aliases

def load_layer(path: Path):
    gdf = gpd.read_file(path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf

def to_crs_safe(gdf, target_crs):
    if gdf.crs is None and target_crs is not None:
        # Все слои в исходном проекте должны быть в одной системе координат.
        # Если это не так, результат будет неверным — проверьте CRS вручную.
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)

def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)
    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    rows = []
    max_row = len(ys) - 1
    cell_area = cell_size * cell_size

    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if not prepared_mask.intersects(geom):
                continue
            inter_area = geom.intersection(mask_union).area
            mask_fraction = inter_area / cell_area if cell_area > 0 else 0
            if mask_fraction <= 0:
                continue
            rows.append({
                "cell_id": len(rows),
                "row": max_row - r,
                "col": c,
                "x": x + cell_size / 2,
                "y": y + cell_size / 2,
                "mask_fraction": mask_fraction,
                "geometry": geom,
            })
    grid = gpd.GeoDataFrame(rows, geometry="geometry", crs=mask.crs)
    shape = (int(grid["row"].max()) + 1, int(grid["col"].max()) + 1)
    return grid, mask_union, shape

def add_distance_feature(grid, source, name):
    src = source[source.geometry.notnull() & ~source.geometry.is_empty]
    if len(src) == 0:
        grid[name] = np.nan
        return grid
    geom = unary_union(src.geometry)
    centroids = grid.geometry.centroid
    grid[name] = centroids.distance(geom).astype(float)
    return grid

def distance_to_proximity(distance, transform="sqrt", q=0.75):
    d = np.asarray(distance, dtype=float)
    finite = np.isfinite(d)
    if finite.sum() == 0:
        return np.zeros_like(d, dtype=float)
    if transform == "sqrt":
        t = np.sqrt(np.clip(d, 0, None))
    elif transform == "cbrt":
        t = np.cbrt(np.clip(d, 0, None))
    elif transform == "log1p":
        t = np.log1p(np.clip(d, 0, None))
    else:
        t = np.clip(d, 0, None)
    scale = safe_quantile(t[finite], q, default=np.nanmedian(t[finite]))
    if not np.isfinite(scale) or scale <= 0:
        scale = np.nanmax(t[finite]) or 1.0
    return np.clip(np.exp(-t / scale), 0, 1)

def collect_point_layers(mask_crs, aliases, exclude_stems):
    frames = []
    info = []
    for stem, path in aliases.items():
        if stem in exclude_stems:
            continue
        try:
            gdf = load_layer(path)
            gdf = to_crs_safe(gdf, mask_crs)
        except Exception:
            continue
        geom_types = set(gdf.geometry.geom_type.astype(str))
        if not any("Point" in t for t in geom_types):
            continue
        gdf = gdf[gdf.geometry.geom_type.str.contains("Point", na=False)].copy()
        if len(gdf) == 0:
            continue
        gdf["source_layer"] = stem
        frames.append(gdf[["source_layer", "geometry"]].copy())
        info.append({"layer": stem, "n": int(len(gdf)), "geom_types": sorted(geom_types)})
    if frames:
        pts = pd.concat(frames, ignore_index=True)
        pts = gpd.GeoDataFrame(pts, geometry="geometry", crs=mask_crs)
    else:
        pts = gpd.GeoDataFrame(columns=["source_layer", "geometry"], geometry="geometry", crs=mask_crs)
    return pts, info

def make_spatial_groups(grid, block_cells=8):
    bx = (grid["col"] // block_cells).astype(int)
    by = (grid["row"] // block_cells).astype(int)
    return (bx.astype(str) + "_" + by.astype(str)).to_numpy()

def top_area_metrics(df, score_col, target_col="target_eval", area_levels=TOP_AREA_LEVELS):
    out = {}
    d = df[[score_col, target_col]].copy()
    d = d[np.isfinite(d[score_col])]
    d = d.sort_values(score_col, ascending=False)
    total_pos = int(d[target_col].sum())
    n = len(d)
    if n == 0 or total_pos == 0:
        for a in area_levels:
            out[f"recall_top_{int(a*100)}pct"] = None
            out[f"precision_top_{int(a*100)}pct"] = None
            out[f"lift_top_{int(a*100)}pct"] = None
        return out
    base_rate = total_pos / n
    for a in area_levels:
        k = max(1, int(np.ceil(a * n)))
        top = d.head(k)
        pos_top = int(top[target_col].sum())
        precision = pos_top / k
        recall = pos_top / total_pos
        lift = precision / base_rate if base_rate > 0 else np.nan
        out[f"recall_top_{int(a*100)}pct"] = float(recall)
        out[f"precision_top_{int(a*100)}pct"] = float(precision)
        out[f"lift_top_{int(a*100)}pct"] = float(lift)
    return out

def build_model():
    if MODEL_KIND == "ExtraTrees":
        return ExtraTreesClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            min_samples_split=RF_MIN_SAMPLES_SPLIT,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    return RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        min_samples_split=RF_MIN_SAMPLES_SPLIT,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

def assign_points_to_cells(grid, points):
    grid["known_positive"] = 0
    if points is None or len(points) == 0:
        return grid, []
    joined = gpd.sjoin(points, grid[["cell_id", "geometry"]], how="inner", predicate="within")
    positive_cells = sorted(joined["cell_id"].dropna().astype(int).unique().tolist())
    grid.loc[grid["cell_id"].isin(positive_cells), "known_positive"] = 1
    return grid, positive_cells

def distance_to_known_points(grid, points):
    if points is None or len(points) == 0:
        return np.full(len(grid), np.nan)
    pts_union = unary_union(points.geometry)
    return grid.geometry.centroid.distance(pts_union).astype(float).to_numpy()

# =========================================================
# 4. LOAD LAYERS
# =========================================================
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)
print("Слои:", sorted(aliases.keys()))

required = ["svita_new", "fasii", "paleo", "struct", "magm"]
missing = [x for x in required if x not in aliases]
if missing:
    raise FileNotFoundError(f"Не найдены обязательные слои: {missing}. Доступные слои: {sorted(aliases.keys())}")

mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["paleo"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["struct"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["magm"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases.get("glub_raz_nw", aliases.get("glub_r_nw"))), mask.crs)
tect2 = to_crs_safe(load_layer(aliases.get("reg_raz_ne", aliases.get("reg_r_ne"))), mask.crs)

exclude_stems = {"svita_new", "fasii", "paleo", "struct", "magm", "glub_raz_nw", "glub_r_nw", "reg_raz_ne", "reg_r_ne"}
points, point_info = collect_point_layers(mask.crs, aliases, exclude_stems)
print("Точечные слои-кандидаты:", point_info)
print("Всего точек-кандидатов:", len(points))

# =========================================================
# 5. GRID AND FEATURES
# =========================================================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)
print("Ячеек в области прогноза:", len(grid), "grid_shape:", grid_shape)

# Расстояния до факторов
for src, name in [
    (facies, "dist_facies"),
    (paleo, "dist_paleo"),
    (struct, "dist_struct"),
    (magm, "dist_magm"),
    (tect1, "dist_tect1"),
    (tect2, "dist_tect2"),
]:
    grid = add_distance_feature(grid, src, name)

# Преобразованные признаки расстояния
for base in ["facies", "paleo", "struct", "magm", "tect1", "tect2"]:
    d = grid[f"dist_{base}"].to_numpy()
    grid[f"sqrt_dist_{base}"] = robust_normalize_01(np.sqrt(np.clip(d, 0, None)), 0.02, 0.98)
    grid[f"cbrt_dist_{base}"] = robust_normalize_01(np.cbrt(np.clip(d, 0, None)), 0.02, 0.98)
    grid[f"log_dist_{base}"] = robust_normalize_01(np.log1p(np.clip(d, 0, None)), 0.02, 0.98)

# Близость к объектам: чем выше, тем перспективнее по фактору
# Трансформации подобраны по логике методики: cbrt для facies/paleo/tect, sqrt для struct/magm.
grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], "cbrt", Q_FACIES)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], "cbrt", Q_PALEO)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], "sqrt", Q_STRUCT)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], "sqrt", Q_MAGM)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], "cbrt", Q_TECT1)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], "cbrt", Q_TECT2)

# Взаимодействия факторов. Это важнее для ML, чем готовый geo_score.
grid["tect_combo"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = grid["tect_combo"] * grid["prox_magm"]
grid["tect_struct_intersection"] = grid["tect_combo"] * grid["prox_struct"]
grid["paleo_struct_intersection"] = grid["prox_paleo"] * grid["prox_struct"]
grid["facies_paleo_intersection"] = grid["prox_facies"] * grid["prox_paleo"]
grid["magm_struct_intersection"] = grid["prox_magm"] * grid["prox_struct"]
grid["all_factors_soft"] = np.power(
    np.clip(grid["prox_facies"] * grid["prox_paleo"] * grid["prox_struct"] * grid["tect_combo"], 0, 1),
    0.25,
)

# Expert baseline: НЕ используется в X. Только для сравнения и выбора осторожных pseudo-negatives.
grid["geo_score_raw"] = (
    0.14 * grid["prox_tect1"] +
    0.14 * grid["prox_tect2"] +
    0.14 * grid["prox_paleo"] +
    0.11 * grid["prox_struct"] +
    0.08 * grid["prox_facies"] +
    0.08 * grid["prox_magm"] +
    0.12 * grid["tect_intersection"] +
    0.10 * grid["tect_struct_intersection"] +
    0.09 * grid["paleo_struct_intersection"] +
    0.10 * grid["tect_magm_intersection"]
)
grid["geo_score"] = robust_normalize_01(grid["geo_score_raw"], 0.02, 0.98)
grid["geo_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "geo_score", grid_shape, passes=1), 0.02, 0.98)

# Coincidence/local — только как небольшая регуляризация финальной поверхности, не как главный сигнал.
grid["coincidence_score"] = robust_normalize_01(
    0.30 * grid["tect_intersection"] +
    0.22 * grid["tect_struct_intersection"] +
    0.20 * grid["tect_magm_intersection"] +
    0.16 * grid["paleo_struct_intersection"] +
    0.12 * grid["facies_paleo_intersection"],
    0.02,
    0.98,
)
grid["local_bonus"] = robust_normalize_01(
    grid["tect_magm_intersection"] * grid["paleo_struct_intersection"] + 0.25 * grid["all_factors_soft"],
    0.02,
    0.98,
)

# =========================================================
# 6. TARGET: POSITIVE + CAREFUL PSEUDO-NEGATIVE
# =========================================================
grid, positive_cells = assign_points_to_cells(grid, points)
grid["dist_to_known_positive"] = distance_to_known_points(grid, points)

# target_eval нужен для проверки: 1 = известная точка, 0 = остальные. Используется только для метрик top-area.
grid["target_eval"] = grid["known_positive"].astype(int)

# Обучающий target строится иначе: positive = известные точки; negative = аккуратно выбранные pseudo-negatives.
grid["train_label"] = np.nan
grid.loc[grid["known_positive"] == 1, "train_label"] = 1

pos_n = int(grid["known_positive"].sum())
print("Positive cells:", pos_n)

if pos_n > 0:
    geo_thr = safe_quantile(grid["geo_score_sm"], NEGATIVE_GEO_QUANTILE, 0.5)
    dist_thr = safe_quantile(grid["dist_to_known_positive"], NEGATIVE_DISTANCE_QUANTILE, 0.0)
    neg_candidates = grid[
        (grid["known_positive"] == 0) &
        (grid["geo_score_sm"] <= geo_thr) &
        (grid["dist_to_known_positive"] >= dist_thr)
    ].copy()
    if MAX_DISTANCE_TO_POSITIVE_FOR_NEGATIVE is not None:
        neg_candidates = neg_candidates[neg_candidates["dist_to_known_positive"] >= MAX_DISTANCE_TO_POSITIVE_FOR_NEGATIVE]

    n_neg = min(len(neg_candidates), max(pos_n * NEGATIVE_TO_POSITIVE_RATIO, pos_n + 1))
    if n_neg > 0:
        sampled_neg = neg_candidates.sample(n=n_neg, random_state=RANDOM_STATE)["cell_id"].tolist()
        grid.loc[grid["cell_id"].isin(sampled_neg), "train_label"] = 0
else:
    sampled_neg = []

train_df = grid[grid["train_label"].notna()].copy()
print("Train positives:", int((train_df["train_label"] == 1).sum()))
print("Train pseudo-negatives:", int((train_df["train_label"] == 0).sum()))

# =========================================================
# 7. FEATURE SET WITHOUT geo_score
# =========================================================
feature_cols = [
    # proximity-факторы
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm", "prox_tect1", "prox_tect2",
    # transformed distances
    "sqrt_dist_facies", "sqrt_dist_paleo", "sqrt_dist_struct", "sqrt_dist_magm", "sqrt_dist_tect1", "sqrt_dist_tect2",
    "cbrt_dist_facies", "cbrt_dist_paleo", "cbrt_dist_struct", "cbrt_dist_magm", "cbrt_dist_tect1", "cbrt_dist_tect2",
    "log_dist_facies", "log_dist_paleo", "log_dist_struct", "log_dist_magm", "log_dist_tect1", "log_dist_tect2",
    # interactions
    "tect_combo", "tect_intersection", "tect_magm_intersection", "tect_struct_intersection",
    "paleo_struct_intersection", "facies_paleo_intersection", "magm_struct_intersection", "all_factors_soft",
    # technical/geometry
    "mask_fraction",
]

# Страховка: geo_score не должен попасть в ML-признаки.
for forbidden in ["geo_score", "geo_score_sm", "geo_score_raw", "prognoz"]:
    assert forbidden not in feature_cols, f"{forbidden} нельзя использовать как ML-признак"

for col in feature_cols:
    grid[col] = pd.to_numeric(grid[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    med = grid[col].median()
    grid[col] = grid[col].fillna(med if np.isfinite(med) else 0.0)

# =========================================================
# 8. SPATIAL CROSS-VALIDATION AND TRAINING
# =========================================================
metrics = {
    "base_dir": str(BASE_DIR),
    "n_cells": int(len(grid)),
    "n_points": int(len(points)),
    "positive_cells": int(pos_n),
    "train_cells": int(len(train_df)),
    "feature_cols": feature_cols,
    "model_kind": MODEL_KIND,
    "spatial_block_cells": int(SPATIAL_BLOCK_CELLS),
}

can_train_ml = (
    len(train_df) >= max(30, MIN_POSITIVE_CELLS * 2) and
    int((train_df["train_label"] == 1).sum()) >= MIN_POSITIVE_CELLS and
    int((train_df["train_label"] == 0).sum()) >= MIN_POSITIVE_CELLS
)

grid["ml_score_raw"] = np.nan
grid["ml_score"] = np.nan
grid["ml_score_oof"] = np.nan

if can_train_ml:
    X_train_all = train_df[feature_cols].to_numpy(dtype=float)
    y_train_all = train_df["train_label"].astype(int).to_numpy()
    groups_all = make_spatial_groups(train_df, SPATIAL_BLOCK_CELLS)
    unique_groups = np.unique(groups_all)
    n_splits = min(N_SPLITS, len(unique_groups))
    n_splits = max(2, n_splits)

    oof = np.full(len(train_df), np.nan, dtype=float)
    fold_metrics = []

    gkf = GroupKFold(n_splits=n_splits)
    for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_train_all, y_train_all, groups_all), start=1):
        # Если в тесте только один класс, метрики ROC/AP невозможны, но прогноз всё равно считаем.
        model = build_model()
        model.fit(X_train_all[tr_idx], y_train_all[tr_idx])
        pred = model.predict_proba(X_train_all[te_idx])[:, 1]
        oof[te_idx] = pred
        fm = {"fold": fold, "n_test": int(len(te_idx)), "test_pos": int(y_train_all[te_idx].sum())}
        if len(np.unique(y_train_all[te_idx])) > 1:
            fm["roc_auc"] = float(roc_auc_score(y_train_all[te_idx], pred))
            fm["average_precision"] = float(average_precision_score(y_train_all[te_idx], pred))
        fold_metrics.append(fm)

    train_df = train_df.copy()
    train_df["ml_score_oof"] = oof
    grid.loc[train_df.index, "ml_score_oof"] = oof

    valid_oof = np.isfinite(oof)
    if valid_oof.sum() > 0 and len(np.unique(y_train_all[valid_oof])) > 1:
        metrics["oof_roc_auc"] = float(roc_auc_score(y_train_all[valid_oof], oof[valid_oof]))
        metrics["oof_average_precision"] = float(average_precision_score(y_train_all[valid_oof], oof[valid_oof]))
    metrics["fold_metrics"] = fold_metrics

    # Финальная модель на всех обучающих positive + pseudo-negative.
    final_model = build_model()
    final_model.fit(X_train_all, y_train_all)
    grid["ml_score_raw"] = final_model.predict_proba(grid[feature_cols].to_numpy(dtype=float))[:, 1]
    grid["ml_score"] = robust_normalize_01(grid["ml_score_raw"], 0.02, 0.98)
    grid["ml_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "ml_score", grid_shape, passes=1), 0.02, 0.98)

    # Feature importance
    importances = pd.DataFrame({
        "feature": feature_cols,
        "importance": final_model.feature_importances_,
    }).sort_values("importance", ascending=False)
    metrics["feature_importance"] = importances.to_dict(orient="records")

    # Baseline metrics against known points on full grid top-area.
    metrics["top_area_ml"] = top_area_metrics(grid, "ml_score_sm", "target_eval", TOP_AREA_LEVELS)
    metrics["top_area_geo_baseline"] = top_area_metrics(grid, "geo_score_sm", "target_eval", TOP_AREA_LEVELS)

    print("ML trained successfully")
    print("OOF ROC-AUC:", metrics.get("oof_roc_auc"))
    print("OOF Average Precision:", metrics.get("oof_average_precision"))
else:
    msg = "Слишком мало положительных/отрицательных обучающих ячеек для честного ML."
    print(msg)
    metrics["ml_warning"] = msg
    if ALLOW_EXPERT_FALLBACK_IF_TOO_FEW_POINTS:
        grid["ml_score"] = grid["geo_score_sm"].copy()
        grid["ml_score_sm"] = grid["geo_score_sm"].copy()
        metrics["fallback"] = "expert_geo_score_sm"
    else:
        grid["ml_score"] = 0.5
        grid["ml_score_sm"] = 0.5
        metrics["fallback"] = "constant_0.5"
    importances = pd.DataFrame(columns=["feature", "importance"])

# =========================================================
# 9. FINAL ML-BASED PROSPECTIVITY
# =========================================================
# ML — главный слой. coincidence/local — небольшая геологическая регуляризация.
grid["ml_prospectivity_raw"] = (
    FINAL_W_ML * grid["ml_score_sm"] +
    FINAL_W_COINCIDENCE * grid["coincidence_score"] +
    FINAL_W_LOCAL * grid["local_bonus"]
)
grid["ml_prospectivity"] = robust_normalize_01(grid["ml_prospectivity_raw"], 0.02, 0.98)

# Совместимость с методикой: меньше prognoz => выше перспективность.
grid["prognoz"] = 1.0 - grid["ml_prospectivity"]

# Классы перспективности
q95 = safe_quantile(grid["ml_prospectivity"], 0.95, 0.95)
q90 = safe_quantile(grid["ml_prospectivity"], 0.90, 0.90)
q80 = safe_quantile(grid["ml_prospectivity"], 0.80, 0.80)
grid["prospect_class"] = "low"
grid.loc[grid["ml_prospectivity"] >= q80, "prospect_class"] = "moderate"
grid.loc[grid["ml_prospectivity"] >= q90, "prospect_class"] = "high"
grid.loc[grid["ml_prospectivity"] >= q95, "prospect_class"] = "very_high"

metrics["final_weights"] = {
    "FINAL_W_ML": FINAL_W_ML,
    "FINAL_W_COINCIDENCE": FINAL_W_COINCIDENCE,
    "FINAL_W_LOCAL": FINAL_W_LOCAL,
}
metrics["top_area_final"] = top_area_metrics(grid, "ml_prospectivity", "target_eval", TOP_AREA_LEVELS)

# =========================================================
# 10. SAVE RESULTS
# =========================================================
# Удаляем старый GPKG, если существует.
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")
if len(points) > 0:
    points.to_file(OUT_GPKG, layer="known_points_candidate", driver="GPKG")

csv_df = grid.drop(columns="geometry").copy()
csv_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

# =========================================================
# 11. PLOTS
# =========================================================
def plot_score(gdf, column, title, path, points=None):
    fig, ax = plt.subplots(figsize=(8, 10))
    gdf.plot(column=column, ax=ax, cmap="viridis", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.8)
    if points is not None and len(points) > 0:
        points.plot(ax=ax, color="red", markersize=8, alpha=0.8)
    ax.set_title(title)
    ax.set_axis_off()
    plt.tight_layout()
    fig.savefig(path, dpi=220)
    plt.show()

plot_score(grid, "ml_prospectivity", "ML prospectivity: итоговый прогноз", OUT_PNG, points if len(points) else None)

# Сравнение baseline vs ML
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, col, title in zip(
    axes,
    ["geo_score_sm", "ml_score_sm", "ml_prospectivity"],
    ["Expert baseline geo_score", "ML score", "Final ML prospectivity"],
):
    grid.plot(column=col, ax=ax, cmap="viridis", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.6)
    if len(points) > 0:
        points.plot(ax=ax, color="red", markersize=5, alpha=0.75)
    ax.set_title(title)
    ax.set_axis_off()
plt.tight_layout()
fig.savefig(OUT_COMPARE, dpi=220)
plt.show()

# Feature importance
if len(importances) > 0:
    top_imp = importances.head(20).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(top_imp["feature"], top_imp["importance"])
    ax.set_title("Top-20 feature importance")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    fig.savefig(OUT_IMPORTANCE, dpi=220)
    plt.show()

print("\nГотово.")
print("GeoPackage:", OUT_GPKG)
print("CSV:", OUT_CSV)
print("Metrics:", OUT_JSON)
print("Map:", OUT_PNG)
print("Compare:", OUT_COMPARE)


BASE_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз
SHP_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\shp_dbf
OUT_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\ml_forecast_result
Слои: ['dayki_buf', 'fasii', 'glub_r_nw', 'glub_raz_nw', 'gr_dol_vp_poly', 'kory', 'layer_00', 'layer_01', 'layer_02', 'result', 'svita_new']


FileNotFoundError: Не найдены обязательные слои: ['paleo', 'struct', 'magm']. Доступные слои: ['dayki_buf', 'fasii', 'glub_r_nw', 'glub_raz_nw', 'gr_dol_vp_poly', 'kory', 'layer_00', 'layer_01', 'layer_02', 'result', 'svita_new']